In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [3]:
target_stations = ['624', '328', '628', '610', '164', '318', '611', '607']
chunk_size = 500000
filtered_chunks = []

for chunk in pd.read_csv('MTA_Subway_Hourly_Ridership.csv',
                         chunksize=chunk_size,
                         dtype={'station_complex_id': str}):
    filtered_chunk = chunk[
        (chunk['borough'] == 'Manhattan') &
        (chunk['transit_mode'] == 'subway') &
        (chunk['station_complex_id'].isin(target_stations))
    ]
    filtered_chunks.append(filtered_chunk)

mta_data = pd.concat(filtered_chunks)
print(f"Filtered rows: {len(mta_data)}")

FileNotFoundError: [Errno 2] No such file or directory: 'MTA_Subway_Hourly_Ridership.csv'

In [ ]:
mta_data.head()

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,Georeference
61,05/13/2023 03:00:00 PM,subway,328,WTC Cortlandt (1),Manhattan,metrocard,Metrocard - Full Fare,184,0,40.711834,-74.01219,POINT (-74.01219 40.711834)
100,05/13/2023 06:00:00 PM,subway,624,"Chambers St (A,C)/WTC (E)/Park Pl (2,3)/Cortla...",Manhattan,metrocard,Metrocard - Other,45,0,40.714110,-74.00858,POINT (-74.00858 40.71411)
109,05/13/2023 10:00:00 PM,subway,328,WTC Cortlandt (1),Manhattan,omny,OMNY - Full Fare,68,0,40.711834,-74.01219,POINT (-74.01219 40.711834)
146,05/13/2023 10:00:00 AM,subway,628,"Fulton St (A,C,J,Z,2,3,4,5)",Manhattan,metrocard,Metrocard - Other,43,0,40.710373,-74.00657,POINT (-74.00657 40.710373)
165,05/13/2023 02:00:00 PM,subway,328,WTC Cortlandt (1),Manhattan,metrocard,Metrocard - Unlimited 7-Day,43,0,40.711834,-74.01219,POINT (-74.01219 40.711834)


In [ ]:
mta_data['transit_timestamp'] = pd.to_datetime(
    mta_data['transit_timestamp'],
    format='%m/%d/%Y %I:%M:%S %p'
)

mta_data['transit_timestamp'] = mta_data['transit_timestamp'].dt.floor('h')
hourly_grouped = mta_data.groupby(['transit_timestamp', 'station_complex_id'])['ridership'].sum().reset_index()

print(f"Aggregated rows: {len(hourly_grouped)}")

Aggregated rows: 141199


In [ ]:
len(hourly_grouped)

141199

In [ ]:
hourly_grouped = hourly_grouped.set_index('transit_timestamp')
hourly_continuous = hourly_grouped.groupby('station_complex_id')['ridership'].resample('h').sum().reset_index()


In [ ]:
hourly_continuous['hour'] = hourly_continuous['transit_timestamp'].dt.hour
hourly_continuous['day_of_week'] = hourly_continuous['transit_timestamp'].dt.dayofweek

In [ ]:
hourly_continuous['hour_sin'] = np.sin(2 * np.pi * hourly_continuous['hour'] / 24.0)
hourly_continuous['hour_cos'] = np.cos(2 * np.pi * hourly_continuous['hour'] / 24.0)
hourly_continuous['day_sin'] = np.sin(2 * np.pi * hourly_continuous['day_of_week'] / 7.0)
hourly_continuous['day_cos'] = np.cos(2 * np.pi * hourly_continuous['day_of_week'] / 7.0)

In [ ]:
hourly_continuous.head()

,station_complex_id,transit_timestamp,ridership,hour,day_of_week,hour_sin,hour_cos,day_sin,day_cos
0,164,2022-05-15 09:00:00,542,9,6,7.071068e-01,-0.707107,-0.781831,0.62349
1,164,2022-05-15 10:00:00,869,10,6,5.000000e-01,-0.866025,-0.781831,0.62349
2,164,2022-05-15 11:00:00,1012,11,6,2.588190e-01,-0.965926,-0.781831,0.62349
3,164,2022-05-15 12:00:00,959,12,6,1.224647e-16,-1.000000,-0.781831,0.62349
4,164,2022-05-15 13:00:00,1153,13,6,-2.588190e-01,-0.965926,-0.781831,0.62349


In [ ]:
hourly_continuous = hourly_continuous.sort_values(by=['station_complex_id', 'transit_timestamp'])

In [ ]:
hourly_continuous['lag_1h'] = hourly_continuous.groupby('station_complex_id')['ridership'].shift(1)
hourly_continuous['lag_24h'] = hourly_continuous.groupby('station_complex_id')['ridership'].shift(24)

hourly_continuous = hourly_continuous.dropna(subset=['lag_1h', 'lag_24h'])
print(f"Final fully processed rows: {len(hourly_continuous)}")

Final fully processed rows: 140928


In [ ]:
hourly_continuous.to_csv('mta_hourly_with_lags.csv', index=False)
print("Data successfully exported to 'mta_hourly_with_lags.csv'")

Data successfully exported to 'mta_hourly_with_lags.csv'
